# 🔍 헷갈리는 BeautifulSoup 개념 완전정복

> 이 노트북은 **자주 혼동되는 개념들**만 모아서 집중적으로 연습합니다.
> 한 셀씩 직접 실행하면서 결과를 눈으로 확인하세요!

---

## 📋 이 노트북에서 다루는 것들

| 번호 | 주제 | 왜 헷갈리나? |
|------|------|------------|
| 1 | `el.name` vs `.select('.name')` | 둘 다 'name'이 들어가 있어서 |
| 2 | `.text` vs `.string` vs `.get_text()` | 비슷해 보이는데 다름 |
| 3 | `find()` vs `select_one()` | 같은 결과처럼 보이는데 다름 |
| 4 | `['class']` vs `.get('class')` vs `.select('.클래스')` | 'class'가 세 곳에 등장 |
| 5 | `['href']` vs `.get('href')` | 왜 두 가지가 있나? |
| 6 | `find_all()` vs `select()` | 같아 보이는데 차이가 있음 |
| 7 | 부모/자식/형제 태그 탐색 | 개념 자체가 낯섦 |


---
## 1️⃣ `el.name` vs `.select('.name')`
### "둘 다 name인데 전혀 달라요!"

많은 분들이 이렇게 헷갈립니다:
```python
soup.select('.name')   # ← 이게 el.name이랑 같은 건가요?
el.name                # ← 이게 class="name"을 가져오는 건가요?
```

**정답: 완전히 다른 것입니다!**

```
soup.select('.name')
  → CSS 선택자로 "class가 name인 요소들"을 찾는 것
  → .name은 선택자 문자열 안에 있는 것

el.name
  → BeautifulSoup 태그 객체의 속성
  → "이 태그의 이름(종류)이 뭐냐?"를 묻는 것
  → 결과: 'div', 'span', 'p', 'a' 등
```


In [ ]:
from bs4 import BeautifulSoup

html = """
<html>
<body>
  <div class="product-card">
    <span class="name">라네즈 수분크림</span>
    <p class="price">38,000원</p>
    <a href="/products/1" class="buy-link">구매</a>
  </div>
</body>
</html>
"""

soup = BeautifulSoup(html, 'html.parser')

# ─── el.name: 태그의 종류를 확인하는 것 ───────────────────────

div_el = soup.find('div')
span_el = soup.find('span')
a_el = soup.find('a')

print("=== el.name: 태그 이름(종류) 확인 ===")
print(f"div 태그의 .name:  '{div_el.name}'")   # 'div'
print(f"span 태그의 .name: '{span_el.name}'")  # 'span'
print(f"a 태그의 .name:    '{a_el.name}'")     # 'a'
print()

# ─── select('.name'): class='name'인 요소를 찾는 것 ──────────

found = soup.select('.name')  # class="name"인 요소 찾기
print("=== select('.name'): class가 name인 요소 찾기 ===")
print(f"찾은 요소: {found}")
print(f"찾은 요소 수: {len(found)}")
print(f"그 요소의 텍스트: '{found[0].text}'")
print()

# ─── 조합해서 쓰면? ──────────────────────────────────────────

name_el = soup.select_one('.name')  # class='name'인 요소 하나
print("=== 조합 사용 ===")
print(f"select_one('.name')로 찾은 요소의 .name: '{name_el.name}'")
# → class='name'인 span 태그를 찾았고, 그 태그의 이름은 'span'


### 정리

```python
el.name              # 태그의 종류: 'div', 'span', 'p', 'a' 등
                     # ↑ el은 BeautifulSoup 태그 객체

soup.select('.name') # class="name"인 요소들 찾기
                     # ↑ .name은 CSS 선택자 문자열의 일부
```

> 💡 **외우는 법**: `el.name`은 "이 태그, 너 뭐야?" 라고 물어보는 것.
> `select('.name')`은 "class가 name인 얘 어딨어?" 라고 찾는 것.

### 언제 `el.name`을 실제로 쓰나요?

```python
# 여러 종류의 태그가 섞여있을 때, 어떤 태그인지 확인할 때 씀
for el in soup.find_all(['h1', 'h2', 'h3']):
    print(f"태그: {el.name}, 텍스트: {el.text.strip()}")
```


In [ ]:
# el.name 실제 활용 예시: 태그 종류 확인하기

html2 = """
<article class="post">
  <h1>크롤링 완전 정복</h1>
  <h2>BeautifulSoup 편</h2>
  <h3>핵심 개념 1</h3>
  <p>본문 내용입니다.</p>
  <h3>핵심 개념 2</h3>
  <p>두 번째 본문입니다.</p>
</article>
"""

soup2 = BeautifulSoup(html2, 'html.parser')

print("=== el.name으로 태그 종류 구분하기 ===")
for el in soup2.find_all(['h1', 'h2', 'h3', 'p']):
    level = el.name          # 'h1', 'h2', 'h3', 'p' 중 하나
    text  = el.text.strip()
    print(f"[{level}] {text}")


---
## 2️⃣ `.text` vs `.string` vs `.get_text()`
### "텍스트 뽑는 방법이 왜 세 가지나 있나요?"

세 가지 모두 텍스트를 가져오지만, **동작이 다른 상황이 있습니다.**


In [ ]:
from bs4 import BeautifulSoup

# 케이스 1: 태그 안에 텍스트만 있는 경우
html_simple = '<span class="price">38,000원</span>'
soup_s = BeautifulSoup(html_simple, 'html.parser')
el = soup_s.find('span')

print("=== 케이스 1: 단순한 경우 (세 가지 모두 같음) ===")
print(f".text:       '{el.text}'")
print(f".string:     '{el.string}'")
print(f".get_text(): '{el.get_text()}'")


In [ ]:
# 케이스 2: 태그 안에 다른 태그가 있는 경우
html_nested = """
<div class="product">
  <span class="brand">라네즈</span>
  <span class="name">수분크림</span>
  <p class="price">38,000원</p>
</div>
"""

soup_n = BeautifulSoup(html_nested, 'html.parser')
div_el = soup_n.find('div')

print("=== 케이스 2: 자식 태그가 있는 경우 ===")

# .text — 모든 자식 태그의 텍스트를 합쳐서 반환
print(f".text:         '{div_el.text}'")
# → '\n  라네즈\n  수분크림\n  38,000원\n'

# .string — 자식이 하나뿐일 때만 동작, 여러 자식이면 None!
print(f".string:       '{div_el.string}'")
# → None (자식이 여러 개라서!)

# .get_text() — .text와 같지만 옵션 지정 가능
print(f".get_text():   '{div_el.get_text()}'")
print(f".get_text(strip=True):  '{div_el.get_text(strip=True)}'")
# strip=True: 각 조각의 앞뒤 공백 제거
print(f".get_text(separator=', '): '{div_el.get_text(separator=', ', strip=True)}'")
# separator: 조각 사이 구분자 지정


In [ ]:
# 케이스 3: 실제 크롤링에서 어떻게 쓰나?

html_product = """
<li class="product-item">
  <strong class="brand">이니스프리</strong>
  <span class="name">  그린티 씨드 세럼  </span>
  <p class="price">
    <del class="original">25,000원</del>
    <strong class="sale">19,900원</strong>
  </p>
</li>
"""

soup_p = BeautifulSoup(html_product, 'html.parser')

# ✅ 실전에서 주로 쓰는 방법: 특정 요소 하나를 골라서 .text.strip()
brand = soup_p.select_one('.brand').text.strip()
name  = soup_p.select_one('.name').text.strip()
sale  = soup_p.select_one('.sale').text.strip()

print(f"브랜드: {brand}")
print(f"상품명: {name}")
print(f"할인가: {sale}")

# ⚠️ 전체 div의 .text는 불필요한 텍스트가 섞여서 보통 안 씀
item = soup_p.find('li')
print(f"\nli 전체 .text: '{item.text}'")
print(f"li 전체 .get_text(strip=True, separator=' | '): '{item.get_text(strip=True, separator=' | ')}'")


### 정리

| 방법 | 언제 씀 | 자식 태그 있을 때 |
|------|---------|----------------|
| `.text` | 가장 많이 씀 | 전체 텍스트 합쳐서 반환 |
| `.string` | 잘 안 씀 | 자식이 여러 개면 `None` |
| `.get_text(strip=True)` | 공백 정리할 때 | 옵션 지정 가능 |

**결론: 대부분 `.text.strip()` 하나면 충분합니다!**


---
## 3️⃣ `find()` vs `select_one()` — 거의 같지만 차이가 있어요


In [ ]:
from bs4 import BeautifulSoup

html = """
<ul class="list">
  <li class="item active">첫 번째</li>
  <li class="item">두 번째</li>
  <li class="item">세 번째</li>
</ul>
"""

soup = BeautifulSoup(html, 'html.parser')

# ─── 같은 결과 ──────────────────────────────────────────────

a1 = soup.find('li', class_='item')
a2 = soup.select_one('li.item')
print("=== 같은 결과인 경우 ===")
print(f"find():        '{a1.text}'")
print(f"select_one():  '{a2.text}'")
# 둘 다 '첫 번째'
print()

# ─── 차이: select_one()이 더 표현력이 풍부 ─────────────────

# find()로는 복합 조건이 어색함
# select_one()은 CSS 선택자 그대로 사용 가능
active = soup.select_one('li.item.active')  # item이면서 active인 것
print("=== select_one()의 강점 ===")
print(f"li.item.active: '{active.text}'")   # '첫 번째'

# find()도 가능하지만 문법이 다름
active2 = soup.find('li', class_=['item', 'active'])
# ↑ 이건 item 또는 active, 의도와 다를 수 있어서 주의!
print()

# ─── 둘 다 못 찾으면 None ─────────────────────────────────

not_found_1 = soup.find('div', class_='nothing')
not_found_2 = soup.select_one('div.nothing')
print("=== 못 찾으면 ===")
print(f"find() 결과:        {not_found_1}")  # None
print(f"select_one() 결과:  {not_found_2}")  # None


### 정리

```python
# 기능상 거의 동일, 실전에서는 select_one()을 더 많이 씀
soup.find('div', class_='name')   →  soup.select_one('div.name')
soup.find('a', id='main')         →  soup.select_one('a#main')

# select_one()이 더 직관적인 경우 (복합 클래스)
soup.select_one('li.item.active')   # item이면서 active
soup.select_one('div > p.price')    # div 바로 아래 p.price
soup.select_one('a[href*="product"]')  # href에 product 포함
```

> 💡 **권장**: 개발자도구에서 Copy selector 하면 CSS 형식으로 나와요.
> 그걸 그대로 `select_one()`에 쓰면 됩니다. `find()`보다 편해요!


---
## 4️⃣ class 관련 3가지 헷갈리는 것

```python
el['class']            # ① HTML 속성 'class'의 값 가져오기
el.get('class')        # ② 위와 같지만 없어도 에러 안 남
soup.select('.name')   # ③ class='name'인 요소 찾기 (탐색!)
```

완전히 다른 용도입니다!


In [ ]:
from bs4 import BeautifulSoup

html = """
<div class="product-card featured new-arrival">
  <span class="name">에어팟 프로 2세대</span>
</div>
<div class="product-card">
  <span class="name">아이폰 15</span>
</div>
"""

soup = BeautifulSoup(html, 'html.parser')

# ① el['class'] — 그 요소의 class 속성값 (리스트로 반환!)
first_div = soup.find('div')
print("=== ① el['class'] — class 속성값 가져오기 ===")
print(f"first_div['class']: {first_div['class']}")
# → ['product-card', 'featured', 'new-arrival']
# ↑ class가 여러 개면 리스트로 반환된다는 점 주의!
print()

# ② el.get('class') — ①과 같지만 없어도 에러 안 남 (안전)
print("=== ② el.get('class') — 안전하게 가져오기 ===")
print(f"first_div.get('class'):  {first_div.get('class')}")
# → ['product-card', 'featured', 'new-arrival']

# 없는 속성이면?
no_id = first_div['id']    if 'id' in first_div.attrs else '없음(KeyError 방지)'
safe  = first_div.get('id', '없음')
print(f"없는 속성 ['id']:       '{no_id}'")
print(f"없는 속성 .get('id'):   '{safe}'")
print()

# ③ soup.select('.product-card') — class='product-card'인 요소 탐색
print("=== ③ select('.product-card') — 해당 클래스 요소 찾기 ===")
cards = soup.select('.product-card')
print(f"찾은 div 수: {len(cards)}")
for card in cards:
    classes = card.get('class')
    name    = card.select_one('.name').text
    print(f"  클래스목록: {classes} / 상품명: {name}")


### 정리

```
el['class']            → 이 요소의 class 속성을 가져와라 (값 조회)
el.get('class')        → 위와 같지만 없어도 None 반환 (안전)
soup.select('.name')   → class="name"인 요소를 찾아라 (탐색)
```

**class는 여러 개일 수 있어서 리스트로 반환됩니다!**
```python
# <div class="card featured hot">
el['class']   # → ['card', 'featured', 'hot']
el['class'][0]  # → 'card' (첫 번째 클래스)

# 특정 클래스가 있는지 확인
if 'featured' in el.get('class', []):
    print("이 상품은 추천 상품!")
```


---
## 5️⃣ `el['href']` vs `el.get('href')`
### "왜 두 가지 방법이 있나요?"


In [ ]:
from bs4 import BeautifulSoup

html = """
<ul>
  <li><a href="/products/1">상품 상세 링크</a></li>
  <li><a>href 없는 링크</a></li>
  <li><a href="https://example.com" target="_blank">외부 링크</a></li>
</ul>
"""

soup = BeautifulSoup(html, 'html.parser')
links = soup.find_all('a')

print("=== 모든 a 태그의 href 확인 ===")
print()

for i, link in enumerate(links, 1):
    print(f"--- {i}번째 a 태그: {link} ---")

    # ❌ el['href'] — href 없으면 KeyError 에러 발생!
    try:
        href1 = link['href']
        print(f"  ['href']:      '{href1}'")
    except KeyError:
        print(f"  ['href']:      ❌ KeyError 에러! href 속성이 없어요.")

    # ✅ el.get('href') — href 없으면 None 반환 (안전)
    href2 = link.get('href')
    print(f"  .get('href'):  {href2}")

    # ✅ el.get('href', '') — 없으면 기본값 반환
    href3 = link.get('href', '링크없음')
    print(f"  .get('href', '링크없음'): '{href3}'")
    print()


In [ ]:
# 실전에서는 이렇게 써요

from bs4 import BeautifulSoup

html = """
<div class="product-list">
  <a href="/products/101" class="product-link">아이폰 15</a>
  <a class="product-link">링크없는상품</a>
  <a href="/products/103" class="product-link">맥북 에어</a>
</div>
"""

soup = BeautifulSoup(html, 'html.parser')
BASE_URL = 'https://www.shopping.co.kr'

print("=== 실전 href 수집 패턴 ===")
for link in soup.select('a.product-link'):
    name = link.text.strip()
    href = link.get('href')  # 없으면 None

    if href:
        full_url = BASE_URL + href  # 상대경로 → 절대경로
    else:
        full_url = None

    print(f"상품명: {name:10}  링크: {full_url}")


---
## 6️⃣ `find_all()` vs `select()` — 비슷하지만 다릅니다


In [ ]:
from bs4 import BeautifulSoup

html = """
<ul class="job-list">
  <li class="job-item urgent">
    <a href="/job/1">데이터 분석가 채용</a>
    <span class="company">네이버</span>
    <span class="deadline urgent">내일 마감</span>
  </li>
  <li class="job-item">
    <a href="/job/2">파이썬 개발자</a>
    <span class="company">카카오</span>
    <span class="deadline">2024-12-31</span>
  </li>
  <li class="job-item">
    <a href="/job/3">ML 엔지니어</a>
    <span class="company">라인</span>
    <span class="deadline">채용시마감</span>
  </li>
</ul>
"""

soup = BeautifulSoup(html, 'html.parser')

print("=== find_all() 사용 ===")
# 태그이름 + class 조합
items1 = soup.find_all('li', class_='job-item')
print(f"find_all('li', class_='job-item'): {len(items1)}개")

print()
print("=== select() 사용 ===")
# CSS 선택자
items2 = soup.select('li.job-item')
print(f"select('li.job-item'): {len(items2)}개")

print()
print("=== 결과가 같음? ===")
print(f"첫 번째 요소 같음: {items1[0] == items2[0]}")  # True

print()
print("=== select()가 더 강력한 경우 ===")
# li 바로 아래 a만 (find_all은 이게 어색)
direct_links = soup.select('li.job-item > a')
print(f"li 바로 아래 a: {len(direct_links)}개")
for a in direct_links:
    print(f"  {a.text} → {a.get('href')}")

print()
# 특정 속성 포함
urgent = soup.select('li.job-item.urgent')
print(f"urgent 클래스 동시에 가진 li: {len(urgent)}개")
for u in urgent:
    print(f"  {u.select_one('a').text}")


### 정리

| | `find_all()` | `select()` |
|---|---|---|
| 기본 사용 | `find_all('li', class_='item')` | `select('li.item')` |
| 여러 클래스 동시 조건 | 복잡 | `select('li.item.active')` |
| 자식 관계 지정 | 어려움 | `select('div > p')` |
| 속성 조건 | `attrs={'data-id': '1'}` | `select('[data-id="1"]')` |
| 결과 | 리스트 | 리스트 |

**결론: `select()`가 더 유연합니다. CSS 선택자를 그대로 쓸 수 있어서요.**


---
## 7️⃣ 부모/자식/형제 태그 탐색

트리 구조를 이해하면 더 정확하게 원하는 요소를 찾을 수 있어요.

```html
<div class="card">          ← div의 자식: h3, p
  <h3>제목</h3>            ← h3의 부모: div, 형제: p
  <p>내용</p>              ← p의 부모: div, 형제: h3
</div>
```


In [ ]:
from bs4 import BeautifulSoup

html = """
<section class="products">
  <div class="card">
    <h3 class="title">맥북 에어 M2</h3>
    <p class="price">1,550,000원</p>
    <p class="stock">재고: 3개</p>
    <a href="/buy/1" class="buy-btn">구매</a>
  </div>
  <div class="card">
    <h3 class="title">아이패드 프로</h3>
    <p class="price">1,200,000원</p>
    <p class="stock">재고: 0개 (품절)</p>
    <a href="/buy/2" class="buy-btn">구매</a>
  </div>
</section>
"""

soup = BeautifulSoup(html, 'html.parser')

# 특정 요소 하나 선택
title_el = soup.select_one('h3.title')
print(f"시작점: {title_el}")
print()

# ─── 부모 (parent) ────────────────────────────────────────
parent = title_el.parent
print(f"부모 태그: <{parent.name}> (class: {parent.get('class')})")
print()

# ─── 자식 (children) ──────────────────────────────────────
print("부모의 자식들:")
for child in parent.children:
    # 텍스트 노드(공백 등)도 포함되어 '
'이 나올 수 있어요
    if child.name:  # 태그인 것만 (텍스트 노드 제외)
        print(f"  <{child.name}> {child.text.strip()}")
print()

# ─── 다음 형제 (next_sibling) ────────────────────────────
# 주의: 공백 텍스트도 형제로 취급됨 → find_next_sibling() 사용 권장
price_sibling = title_el.find_next_sibling('p')
print(f"h3 다음 p 형제: {price_sibling.text}")

# 이전 형제
# prev = price_sibling.find_previous_sibling('h3')
# print(f"p 이전 h3 형제: {prev.text}")
print()

# ─── 실전 활용: 부모 타고 올라가기 ──────────────────────
# buy_btn → 그 div 안의 title 찾기 (형제나 부모 활용)
buy_btn = soup.select('.buy-btn')[1]  # 두 번째 구매 버튼
card_div = buy_btn.parent             # 바로 위 div.card
title_in_same_card = card_div.select_one('.title')
print(f"두 번째 구매 버튼과 같은 카드의 상품명: {title_in_same_card.text}")


---
## 🎯 종합 연습 — 모든 개념 한번에 적용

지금까지 배운 개념을 모두 사용해서 데이터를 수집해봅시다.


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

# 앞서 만든 mock_shopping.html을 직접 문자열로 파싱

html_shopping = """
<ol class="product-list">
  <li class="product-card" data-rank="1" data-product-id="P001">
    <div class="rank-badge">1</div>
    <div class="brand">라네즈</div>
    <h3 class="product-name">네오 쿠션 파운데이션 21호</h3>
    <div class="price-area">
      <span class="original-price">38,000원</span>
      <span class="sale-price">27,000원</span>
      <span class="discount-rate">29%</span>
    </div>
    <div class="rating-area">
      <span class="stars">★★★★☆</span>
      <span class="review-count">(리뷰 1,204개)</span>
    </div>
    <a href="/products/P001" class="buy-link">구매하기</a>
  </li>
  <li class="product-card" data-rank="2" data-product-id="P002">
    <div class="rank-badge">2</div>
    <div class="brand">이니스프리</div>
    <h3 class="product-name">수분크림 그린티 씨드 세럼</h3>
    <div class="price-area">
      <span class="original-price">25,000원</span>
      <span class="sale-price">19,900원</span>
      <span class="discount-rate">20%</span>
    </div>
    <div class="rating-area">
      <span class="stars">★★★★★</span>
      <span class="review-count">(리뷰 3,891개)</span>
    </div>
    <a href="/products/P002" class="buy-link">구매하기</a>
  </li>
  <li class="product-card" data-rank="3" data-product-id="P003">
    <div class="rank-badge">3</div>
    <div class="brand">설화수</div>
    <h3 class="product-name">자음 생크림 50ml 리미티드</h3>
    <div class="price-area">
      <span class="original-price">120,000원</span>
      <span class="sale-price">96,000원</span>
      <span class="discount-rate">20%</span>
    </div>
    <div class="rating-area">
      <span class="stars">★★★★★</span>
      <span class="review-count">(리뷰 572개)</span>
    </div>
    <a href="/products/P003" class="buy-link">구매하기</a>
  </li>
</ol>
"""

soup = BeautifulSoup(html_shopping, 'html.parser')

# ─── 데이터 추출 ─────────────────────────────────────────

rows = []
cards = soup.select('li.product-card')

for card in cards:
    # data-rank 속성 (el['속성'] 방식)
    rank = card.get('data-rank')
    pid  = card.get('data-product-id')

    # 브랜드 (.brand의 태그 이름은?)
    brand_el = card.select_one('.brand')
    brand    = brand_el.text.strip() if brand_el else None
    print(f"[탐색] brand_el.name = '{brand_el.name}'")  # 태그 이름 확인!

    # 상품명
    name_el = card.select_one('.product-name')
    name    = name_el.text.strip() if name_el else None

    # 가격 (여러 span 중 sale-price만)
    sale_el = card.select_one('.sale-price')
    sale    = sale_el.text.strip() if sale_el else None

    # 할인율
    disc_el = card.select_one('.discount-rate')
    disc    = disc_el.text.strip() if disc_el else '0%'

    # 리뷰수 (괄호와 '리뷰' 제거: '(리뷰 1,204개)' → '1204')
    review_el = card.select_one('.review-count')
    if review_el:
        import re
        nums = re.sub(r'[^0-9]', '', review_el.text)  # 숫자만
        review_cnt = int(nums) if nums else 0
    else:
        review_cnt = 0

    # 링크 (.get('href') — 안전한 방법)
    link_el = card.select_one('.buy-link')
    href    = link_el.get('href') if link_el else None

    rows.append({
        '순위': rank, '상품ID': pid, '브랜드': brand,
        '상품명': name, '할인가': sale, '할인율': disc,
        '리뷰수': review_cnt, '링크': href,
    })

df = pd.DataFrame(rows)
print()
print(df.to_string(index=False))


## ✅ 이 노트북에서 배운 것 최종 정리

```python
# 헷갈리는 것들 한눈에 보기

el.name                 # 태그 이름: 'div', 'span', 'a' ...
el.text.strip()         # 텍스트 (가장 많이 씀)
el.get_text(strip=True) # 텍스트 (공백 옵션)
el.string               # 자식 하나일 때만, 아니면 None

el['href']              # 속성값 (없으면 KeyError!)
el.get('href')          # 속성값 (없으면 None, 안전)
el.get('href', '기본값') # 속성값 (없으면 기본값)

el['class']             # class 속성 (리스트로 반환!)
el.get('class', [])     # class 속성 (없으면 빈 리스트)

soup.find('태그')        # 하나 (태그/id/class로)
soup.select_one('선택자') # 하나 (CSS 선택자로) ← 주로 이걸 씀
soup.find_all('태그')    # 전부 (리스트)
soup.select('선택자')    # 전부 (CSS 선택자) ← 주로 이걸 씀

el.parent               # 부모 태그
el.find_next_sibling('태그') # 다음 형제 태그
```
